In [ ]:
# Upgrade packaging tools first
!pip install -q -U pip setuptools wheel

# Install the main Hugging Face packages
!pip install -q -U transformers datasets accelerate sentencepiece evaluate

# Install seqeval separately
!pip install -q seqeval==1.2.2

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 22.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 818.2/818.2 kB 26.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.
torch 2.11.0+cpu requires setuptools<82, but you have setuptools 84.0.0 which is incompatible.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done


In [ ]:
import numpy as np
import pandas as pd
import torch

from datasets import load_dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    DataCollatorForTokenClassification,
    TrainingArguments,
    Trainer
)

import evaluate

In [ ]:
print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

GPU available: False


#### Load Dataset

In [ ]:
dataset = load_dataset(
    "ai4privacy/open-pii-masking-500k-ai4privacy"
)

dataset

README.md:   0%|          | 0.00/23.3k [00:00<?, ?B/s]

data/train/train.jsonl: reconstructing file:   0%|          |  0.00B /  566MB            

data/train/train.jsonl: downloading bytes:           |  0.00B            

data/validation/test.jsonl: reconstructing file:   0%|          |  0.00B /  142MB            

data/validation/test.jsonl: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/464150 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/116077 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['source_text', 'masked_text', 'privacy_mask', 'split', 'uid', 'language', 'region', 'script', 'mbert_tokens', 'mbert_token_classes'],
        num_rows: 464150
    })
    validation: Dataset({
        features: ['source_text', 'masked_text', 'privacy_mask', 'split', 'uid', 'language', 'region', 'script', 'mbert_tokens', 'mbert_token_classes'],
        num_rows: 116077
    })
})

In [ ]:
dataset["train"][0]

{'source_text': '20:10:26 Venanzius Höttermann Revés యొక్క వివాహం July/95 నాడు జరిగింది, Tadaలో Faizabad Road వద్ద.',
 'masked_text': '[TIME_1] [GIVENNAME_1] [SURNAME_1] యొక్క వివాహం [DATE_1] నాడు జరిగింది, [CITY_1]లో [STREET_1] వద్ద.',
 'privacy_mask': [{'label': 'TIME',
   'start': 0,
   'end': 8,
   'value': '20:10:26',
   'label_index': 1},
  {'label': 'GIVENNAME',
   'start': 9,
   'end': 18,
   'value': 'Venanzius',
   'label_index': 1},
  {'label': 'SURNAME',
   'start': 19,
   'end': 35,
   'value': 'Höttermann Revés',
   'label_index': 1},
  {'label': 'DATE',
   'start': 49,
   'end': 56,
   'value': 'July/95',
   'label_index': 1},
  {'label': 'CITY', 'start': 72, 'end': 76, 'value': 'Tada', 'label_index': 1},
  {'label': 'STREET',
   'start': 79,
   'end': 92,
   'value': 'Faizabad Road',
   'label_index': 1}],
 'split': 'train',
 'uid': 5387382,
 'language': 'te',
 'region': 'IN',
 'script': 'Telu',
 'mbert_tokens': ['20',
  ':',
  '10',
  ':',
  '26',
  'Ve',
  '##nan',
  

# Training Phase

#### Give a subset of the entire dataset

In [ ]:
SEED = 42

train_dataset = dataset["train"].shuffle(seed=SEED).select(range(30_000))
val_dataset = dataset["validation"].shuffle(seed=SEED).select(range(5_000))

print("Train:", len(train_dataset))
print("Validation:", len(val_dataset))

Train: 30000
Validation: 5000


In [ ]:
example = train_dataset[0]

print("SOURCE:")
print(example["source_text"])

print("\nPRIVACY MASK:")
print(example["privacy_mask"])

SOURCE:
Family history request: searching for Anemone Beselga birth records circa 20/06/1961

PRIVACY MASK:
[{'label': 'GIVENNAME', 'start': 38, 'end': 45, 'value': 'Anemone', 'label_index': 1}, {'label': 'SURNAME', 'start': 46, 'end': 53, 'value': 'Beselga', 'label_index': 1}, {'label': 'DATE', 'start': 74, 'end': 84, 'value': '20/06/1961', 'label_index': 1}]


####  Get All PPI Labels

In [ ]:
all_labels = set()

for example in train_dataset:
    for entity in example["privacy_mask"]:
        all_labels.add(entity["label"])

all_labels = sorted(all_labels)

print("Number of PII classes:", len(all_labels))
print(all_labels)

Number of PII classes: 20
['AGE', 'BUILDINGNUM', 'CITY', 'CREDITCARDNUMBER', 'DATE', 'DRIVERLICENSENUM', 'EMAIL', 'GENDER', 'GIVENNAME', 'IDCARDNUM', 'PASSPORTNUM', 'SEX', 'SOCIALNUM', 'STREET', 'SURNAME', 'TAXNUM', 'TELEPHONENUM', 'TIME', 'TITLE', 'ZIPCODE']


#### Get BIO Labels
* Beginning
* Inside
* Outside

In [ ]:
label_list = ["O"]

for label in all_labels:
    label_list.append(f"B-{label}")
    label_list.append(f"I-{label}")

label2id = {
    label: i
    for i, label in enumerate(label_list)
}

id2label = {
    i: label
    for label, i in label2id.items()
}

print("Number of BIO labels:", len(label_list))
print(label_list)

Number of BIO labels: 41
['O', 'B-AGE', 'I-AGE', 'B-BUILDINGNUM', 'I-BUILDINGNUM', 'B-CITY', 'I-CITY', 'B-CREDITCARDNUMBER', 'I-CREDITCARDNUMBER', 'B-DATE', 'I-DATE', 'B-DRIVERLICENSENUM', 'I-DRIVERLICENSENUM', 'B-EMAIL', 'I-EMAIL', 'B-GENDER', 'I-GENDER', 'B-GIVENNAME', 'I-GIVENNAME', 'B-IDCARDNUM', 'I-IDCARDNUM', 'B-PASSPORTNUM', 'I-PASSPORTNUM', 'B-SEX', 'I-SEX', 'B-SOCIALNUM', 'I-SOCIALNUM', 'B-STREET', 'I-STREET', 'B-SURNAME', 'I-SURNAME', 'B-TAXNUM', 'I-TAXNUM', 'B-TELEPHONENUM', 'I-TELEPHONENUM', 'B-TIME', 'I-TIME', 'B-TITLE', 'I-TITLE', 'B-ZIPCODE', 'I-ZIPCODE']


### Load the XLM-ROBERTa-Model

In [ ]:
MODEL_NAME = "FacebookAI/xlm-roberta-base"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    use_fast=True
)

config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.10M [00:00<?, ?B/s]

#### Convert Character spans  ->  Token Labels

##### One mask in original text can be converted into many tokens dpending on the XLM-Roberta Tokenizer. So, are going over each PII class and check if the indices if tokens
---
##### overlap with the indices of the mask, if yes then if first Token give it "B-{label of that mask in the original text}", else "I-{...}". Rest all tokens will get "O" class.
---
> Example ->
*"Venanzius"*
* start = 9
* end = 18
* label = GIVENNAME

> After Tokenization ->
* Ve
* ##nan
* ##ziu
* ##s
> All four Overlaps
* Ve       → B-GIVENNAME
* ##nan    → I-GIVENNAME
* ##ziu    → I-GIVENNAME
* ##s      → I-GIVENNAME

In [ ]:


def tokenize_and_align_labels(example):
    text = example["source_text"]

    tokenized = tokenizer(
        text,
        truncation=True,
        max_length=256,
        return_offsets_mapping=True     ## Return the Character spans of the tokens in the original texts too
    )

    offsets = tokenized["offset_mapping"]

    labels = ["O"] * len(offsets)

    entities = example["privacy_mask"]

    for entity in entities:
        entity_start = entity["start"]
        entity_end = entity["end"]
        entity_label = entity["label"]

        first_token = True

        for token_idx, (token_start, token_end) in enumerate(offsets):
            # Special tokens
            if token_start == token_end:
                continue

            # No overlap
            if token_end <= entity_start or token_start >= entity_end:
                continue

            # Token overlaps entity
            if first_token:
                labels[token_idx] = f"B-{entity_label}"
                first_token = False
            else:
                labels[token_idx] = f"I-{entity_label}"

    tokenized["labels"] = [
        label2id[label]
        for label in labels
    ]

    # We don't need this after alignment
    tokenized.pop("offset_mapping")

    return tokenized

*{
    "input_ids": [...],
    "attention_mask": [...],
    "labels": [0, 7, 8, 8, 8, 9, 10, ...]
}*

#### Test the alignment

In [ ]:
test_example = train_dataset[0]

processed = tokenize_and_align_labels(test_example)

tokens = tokenizer.convert_ids_to_tokens(
    processed["input_ids"]
)

labels = [
    id2label[label_id]
    for label_id in processed["labels"]
]

for token, label in zip(tokens, labels):
    print(f"{token:20} {label}")

<s>                  O
▁Family              O
▁history             O
▁request             O
:                    O
▁searching           O
▁for                 O
▁Ane                 B-GIVENNAME
mone                 I-GIVENNAME
▁Be                  B-SURNAME
sel                  I-SURNAME
ga                   I-SURNAME
▁birth               O
▁records             O
▁circa               O
▁20                  B-DATE
/06                  I-DATE
/                    I-DATE
1961                 I-DATE
</s>                 O


### Process the datasets

In [ ]:
tokenized_train = train_dataset.map(
    tokenize_and_align_labels,
    remove_columns=train_dataset.column_names,
    desc="Tokenizing training data"
)

tokenized_val = val_dataset.map(
    tokenize_and_align_labels,
    remove_columns=val_dataset.column_names,
    desc="Tokenizing validation data"
)

Tokenizing training data:   0%|          | 0/30000 [00:00<?, ? examples/s]

Tokenizing validation data:   0%|          | 0/5000 [00:00<?, ? examples/s]

In [ ]:
tokenized_train

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 30000
})

In [ ]:
data_collator = DataCollatorForTokenClassification(
    tokenizer=tokenizer
)

#### Load XLM-R for token classification

In [ ]:
model = AutoModelForTokenClassification.from_pretrained(   ## Take the XLM-R encoder and attach a linear CLASSIFICATION NEURAL NETWROK on TOP OF IT
    MODEL_NAME,
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id
)

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.12GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] XLMRobertaForTokenClassification LOAD REPORT from: FacebookAI/xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
classifier.bias             | MISSING    | 
classifier.weight           | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


### Evaluation

In [ ]:
seqeval = evaluate.load("seqeval")

In [ ]:
def compute_metrics(eval_predictions):

    predictions, labels = eval_predictions

    predictions = np.argmax(predictions, axis=2)

    true_predictions = []
    true_labels = []

    for prediction, label in zip(predictions, labels):

        current_predictions = []
        current_labels = []

        for pred, lab in zip(prediction, label):

            # Ignore padding/special tokens
            if lab == -100:
                continue

            current_predictions.append(
                id2label[pred]
            )

            current_labels.append(
                id2label[lab]
            )

        true_predictions.append(current_predictions)
        true_labels.append(current_labels)

    results = seqeval.compute(
        predictions=true_predictions,
        references=true_labels
    )

    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"]
    }

### Training Configurtion

* Batch size = 8
* Gradient accumulation = 2
* Effective batch size = 16
* Epochs = 3
* Learning rate = 2e-5
* FP16 = enabled

#### Actually what is happening in the Training Phase is that ->
---
1. The Normal model has 12 Transformer Encoder layers
2. At the end of the 12 layers, each token of the sentence gets a 768-dimensional vector representing the token with attention wrt to the sentence.
3. Now in the modified model we removed the last MASK LANGUAGE MODELLING Layer and replace with a linear classfication layer.
4. What happens in the classification layer is that it takes the 768-dimensional embedding for each token and the last layer classifies that token.

In [ ]:
training_args = TrainingArguments(
    output_dir="./xlmr-pii",

    eval_strategy="epoch",
    save_strategy="epoch",

    learning_rate=2e-5,

    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,

    gradient_accumulation_steps=2,

    num_train_epochs=3,

    weight_decay=0.01,

    fp16=torch.cuda.is_available(),

    logging_steps=100,

    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,

    save_total_limit=2,

    report_to="none"
)

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,

    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,

    processing_class=tokenizer,

    data_collator=data_collator,

    compute_metrics=compute_metrics
)

In [ ]:
## Train the model
trainer.train()

Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,0.176362,0.073106,0.838800,0.866362,0.852358,0.975571
2,0.106988,0.063718,0.861531,0.883638,0.872445,0.978386
3,0.087711,0.062290,0.877076,0.894309,0.885609,0.980174


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=5625, training_loss=0.20097813347710503, metrics={'train_runtime': 1215.7514, 'train_samples_per_second': 74.028, 'train_steps_per_second': 4.627, 'total_flos': 2303162848976640.0, 'train_loss': 0.20097813347710503, 'epoch': 3.0})

#### Evaluate the model

In [ ]:
results = trainer.evaluate()

results

Training Loss,Validation Loss,Epoch,Precision,Recall,F1,Accuracy
0.087711,0.062290,3,0.877076,0.894309,0.885609,0.980174


{'eval_loss': 0.06229018419981003,
 'eval_precision': 0.8770764119601329,
 'eval_recall': 0.8943089430894309,
 'eval_f1': 0.8856088560885609,
 'eval_accuracy': 0.9801737255896525}

In [ ]:
OUTPUT_DIR = "/content/drive/MyDrive/PII and PHI Models Fine Tune"

trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

print("Saved to:", OUTPUT_DIR)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved to: /content/drive/MyDrive/PII and PHI Models Fine Tune


In [ ]:
from transformers import pipeline

pii_detector = pipeline(
    "token-classification",
    model=OUTPUT_DIR,
    tokenizer=OUTPUT_DIR,
    aggregation_strategy="simple",
    device=0 if torch.cuda.is_available() else -1
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [ ]:
text = """
Patient Rajesh Kumar lives in Surat.
His email is rajesh.kumar@gmail.com.
His phone number is 9876543210.
"""

results = pii_detector(text)

for result in results:
    print(result)

{'entity_group': 'GIVENNAME', 'score': np.float32(0.981905), 'word': 'Rajesh Kumar', 'start': 9, 'end': 21}
{'entity_group': 'CITY', 'score': np.float32(0.9963272), 'word': 'Surat', 'start': 31, 'end': 36}
{'entity_group': 'EMAIL', 'score': np.float32(0.9997109), 'word': 'rajesh.kumar@gmail.com', 'start': 51, 'end': 73}
{'entity_group': 'SOCIALNUM', 'score': np.float32(0.465766), 'word': '9876543210', 'start': 95, 'end': 105}


## SIMPLE MASKING FUNCTION FOR RAG APPLICATION

In [ ]:
def mask_pii(text):

    entities = pii_detector(text)

    # Sort from right to left
    entities = sorted(
        entities,
        key=lambda x: x["start"],
        reverse=True
    )

    masked_text = text

    for entity in entities:

        start = entity["start"]
        end = entity["end"]
        label = entity["entity_group"]

        replacement = f"[{label}]"

        masked_text = (
            masked_text[:start]
            + replacement
            + masked_text[end:]
        )

    return masked_text

In [ ]:
text = """
Patient Rajesh Kumar lives in Surat.
Email: rajesh.kumar@gmail.com
Phone: 9876543210
"""

print(mask_pii(text))


Patient [GIVENNAME] lives in [CITY].
Email: [EMAIL]
Phone: [SOCIALNUM]



# Load the trained Model

In [ ]:
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification
)

MODEL_PATH = "/content/drive/MyDrive/PII and PHI Models Fine Tune"

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)

model = AutoModelForTokenClassification.from_pretrained(
    MODEL_PATH
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [ ]:
model.eval()

XLMRobertaForTokenClassification(
  (dropout): Dropout(p=0.1, inplace=False)
  (classifier): Linear(in_features=768, out_features=41, bias=True)
  (roberta): XLMRobertaModel(
    (embeddings): XLMRobertaEmbeddings(
      (word_embeddings): Embedding(250002, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
      (position_embeddings): Embedding(514, 768, padding_idx=1)
    )
    (encoder): XLMRobertaEncoder(
      (layer): ModuleList(
        (0-11): 12 x XLMRobertaLayer(
          (attention): XLMRobertaAttention(
            (self): XLMRobertaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )


In [ ]:
import torch

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model.to(device)

print("Using:", device)

Using: cpu


In [ ]:
from transformers import pipeline

pii_detector = pipeline(
    "token-classification",
    model=model,
    tokenizer=tokenizer,
    aggregation_strategy="simple",
    device=0 if torch.cuda.is_available() else -1
)

In [ ]:
text = """
Patient Rajesh Kumar lives in Surat.
His email is rajesh.kumar@gmail.com.
His phone number is 9876543210.
"""

results = pii_detector(text)

for result in results:
    print(result)

{'entity_group': 'GIVENNAME', 'score': np.float32(0.9819049), 'word': 'Rajesh Kumar', 'start': 9, 'end': 21}
{'entity_group': 'CITY', 'score': np.float32(0.9963272), 'word': 'Surat', 'start': 31, 'end': 36}
{'entity_group': 'EMAIL', 'score': np.float32(0.9997109), 'word': 'rajesh.kumar@gmail.com', 'start': 51, 'end': 73}
{'entity_group': 'SOCIALNUM', 'score': np.float32(0.46576664), 'word': '9876543210', 'start': 95, 'end': 105}
